# AQI Predictor

This project predicts the AQI for the next 3 days in Karachi, Lahore, Faisalabad, Islamabad and Peshawar. Pollutants come from OpenWeather and weather comes from Open-Meteo. The features are stored in Hopsworks and the forecast is shown on a Streamlit dashboard.

## 00 — Setup

I will set up the constants used in the rest of the notebook: the five cities, the API endpoints and the EPA breakpoint table that turns pollutant readings into an AQI number.

In [ ]:
import os
import pathlib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from tenacity import retry,stop_after_attempt,wait_exponential

load_dotenv()
OPENWEATHER_KEY = os.getenv("OPENWEATHER_API_KEY")
HOPSWORKS_KEY = os.getenv("HOPSWORKS_API_KEY")

data_dir = pathlib.Path("data")
fig_dir = pathlib.Path("outputs/figures")
data_dir.mkdir(parents=True,exist_ok=True)
fig_dir.mkdir(parents=True,exist_ok=True)

sns.set_theme(style="whitegrid")

### Cities

I will use the city centre coordinates for the five cities. Both APIs take a lat and lon so the same dictionary works for both.

In [ ]:
CITIES = {
    "Karachi":(24.86,67.01),
    "Lahore":(31.55,74.35),
    "Faisalabad":(31.41,73.07),
    "Islamabad":(33.72,73.06),
    "Peshawar":(34.01,71.57),
}

### API Endpoints

OpenWeather only sells historical weather on a paid plan so I will take the pollutants from OpenWeather and the weather from Open-Meteo which gives the ERA5 archive for free and without a key. The archive is about five days behind real time so I will use the forecast endpoint with past_days to fill those last few days.

In [ ]:
POLLUTION_URL = "http://api.openweathermap.org/data/2.5/air_pollution"
POLLUTION_HISTORY_URL = "http://api.openweathermap.org/data/2.5/air_pollution/history"
POLLUTION_FORECAST_URL = "http://api.openweathermap.org/data/2.5/air_pollution/forecast"

WEATHER_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

POLLUTANTS = ["pm2_5","pm10","o3","no2","so2","co","no","nh3"]

WEATHER_VARS = [
    "temperature_2m","relative_humidity_2m","dew_point_2m","precipitation",
    "surface_pressure","wind_speed_10m","wind_direction_10m","cloud_cover",
]

### EPA Breakpoints

OpenWeather returns its own AQI but it is only an integer from 1 to 5 which is too coarse to regress on. So I will compute the US EPA AQI instead, the 0 to 500 scale that aqicn and most public dashboards show.

The EPA gives a breakpoint table for each pollutant. Each row maps a concentration range to an AQI range and the AQI is found by interpolating between the two. Each pollutant is also averaged over a set number of hours before the lookup and the four gases have to be converted from µg/m³ first, so I will keep those here too.

In [ ]:
# conc low, conc high, aqi low, aqi high
BREAKPOINTS = {
    # pm2_5 is the 2024 revised table
    "pm2_5":[
        (0.0,9.0,0,50),(9.1,35.4,51,100),(35.5,55.4,101,150),
        (55.5,125.4,151,200),(125.5,225.4,201,300),(225.5,325.4,301,500),
    ],
    "pm10":[
        (0,54,0,50),(55,154,51,100),(155,254,101,150),
        (255,354,151,200),(355,424,201,300),(425,604,301,500),
    ],
    "o3":[
        (0.000,0.054,0,50),(0.055,0.070,51,100),(0.071,0.085,101,150),
        (0.086,0.105,151,200),(0.106,0.200,201,300),
    ],
    "co":[
        (0.0,4.4,0,50),(4.5,9.4,51,100),(9.5,12.4,101,150),
        (12.5,15.4,151,200),(15.5,30.4,201,300),(30.5,50.4,301,500),
    ],
    "so2":[
        (0,35,0,50),(36,75,51,100),(76,185,101,150),
        (186,304,151,200),(305,604,201,300),(605,1004,301,500),
    ],
    "no2":[
        (0,53,0,50),(54,100,51,100),(101,360,101,150),
        (361,649,151,200),(650,1249,201,300),(1250,2049,301,500),
    ],
}

AVERAGING_HOURS = {"pm2_5":24,"pm10":24,"o3":8,"co":8,"so2":1,"no2":1}

# o3 and co need ppm and the other two need ppb
MOLECULAR_WEIGHTS = {"o3":48.0,"co":28.01,"so2":64.06,"no2":46.01}

### AQI Categories

The scale is split into six named bands. I will store each band with its upper limit so I can find the category for any AQI value.

In [ ]:
AQI_CATEGORIES = [
    (50,"Good","green"),
    (100,"Moderate","gold"),
    (150,"Unhealthy for Sensitive Groups","orange"),
    (200,"Unhealthy","red"),
    (300,"Very Unhealthy","purple"),
    (500,"Hazardous","darkred"),
]